# Pydantic V2 - Full Course - Learn the BEST Library for Data Validation and Parsing

https://youtu.be/7aBRk_JP-qY

In [40]:
!pip install pydantic

In [41]:
import pydantic
print(pydantic.__version__)

2.13.4


Validierung ohne Pydantic

In [42]:
class User:
    def __init__(self, id, name='Jane Doe'):
        if not isinstance(id, int):
            raise TypeError(f'Expected id to be an int, got {type(id).__name__}')
        
        if not isinstance(name, str):
            raise TypeError(f'Expected name to be a str, got {type(name).__name__}')
        
        self.id = id
        self.name = name

try:
    user = User(id='123')
except TypeError as e:
    print(e)


Expected id to be an int, got str


Mit Pydantic

In [44]:
from pydantic import BaseModel


class User(BaseModel):
    id: int
    name: str = 'Jane Doe'

In [45]:
user = User(id='123')

In [46]:
print(user.id)

123


In [47]:
print(user.model_fields_set)
user = User(id='123', name='Joe Doe')
print(user.model_fields_set)


{'id'}
{'id', 'name'}


In [9]:
print(user.model_dump())
print(user.model_dump_json())
print(user.model_json_schema())

{'id': 123, 'name': 'Joe Doe'}
{"id":123,"name":"Joe Doe"}
{'properties': {'id': {'title': 'Id', 'type': 'integer'}, 'name': {'default': 'Jane Doe', 'title': 'Name', 'type': 'string'}}, 'required': ['id'], 'title': 'User', 'type': 'object'}


### Nested models

In [10]:
from typing import List, Optional
from pydantic import BaseModel


class Food(BaseModel):
    name: str
    price: float
    ingredients: Optional[List[str]] = None


class Restaurant(BaseModel):
    name: str
    location: str
    foods: List[Food]


restaurant_instance = Restaurant(
    name="Tasty Bites",
    location="123, Flavor Street",
    foods=[
        {"name": "Cheese Pizza", "price": 12.50, "ingredients": ["Cheese", "Tomato Sauce", "Dough"]},
        {"name": "Veggie Burger", "price": 8.99}
    ]
)

print(restaurant_instance)
print(restaurant_instance.model_dump())

name='Tasty Bites' location='123, Flavor Street' foods=[Food(name='Cheese Pizza', price=12.5, ingredients=['Cheese', 'Tomato Sauce', 'Dough']), Food(name='Veggie Burger', price=8.99, ingredients=None)]
{'name': 'Tasty Bites', 'location': '123, Flavor Street', 'foods': [{'name': 'Cheese Pizza', 'price': 12.5, 'ingredients': ['Cheese', 'Tomato Sauce', 'Dough']}, {'name': 'Veggie Burger', 'price': 8.99, 'ingredients': None}]}


In [11]:
!pip install 'pydantic[email]'

  Using cached email_validator-2.3.0-py3-none-any.whl.metadata (26 kB)
  Using cached dnspython-2.8.0-py3-none-any.whl.metadata (5.7 kB)
Using cached email_validator-2.3.0-py3-none-any.whl (35 kB)
Using cached dnspython-2.8.0-py3-none-any.whl (331 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [email-validator]


In [12]:
!pip install email-validator

In [13]:
!pip show email-validator

Name: email-validator
Version: 2.3.0
Summary: A robust email address syntax and deliverability validation library.
Home-page: https://github.com/JoshData/python-email-validator
Author: Joshua Tauberer
Author-email: jt@occams.info
License: Unlicense
Location: /opt/anaconda3/envs/p3135/lib/python3.13/site-packages
Requires: dnspython, idna
Required-by: 


In [14]:
from typing import List
from pydantic import BaseModel, EmailStr, PositiveInt, conlist, Field, HttpUrl

class Address(BaseModel):
    street: str
    city: str
    state: str
    zip_code: str

class Employee(BaseModel):
    name: str
    position: str
    email: EmailStr
    
class Owner(BaseModel):
    name: str
    email: EmailStr
    
class Restaurant(BaseModel):
    name: str = Field(..., pattern=r"^[a-zA-Z0-9-' ]+$")
    owner: Owner
    address: Address
    employees: conlist(Employee, min_length=2)
    number_of_seats: PositiveInt
    delivery: bool
    website: HttpUrl

# Creating an instance of the Restaurant class
restaurant_instance = Restaurant(
    name="Tasty Bites",
    owner={
        "name": "John Doe",
        "email": "john.doe@example.com"
    },
    address={
        "street": "123, Flavor Street",
        "city": "Tastytown",
        "state": "TS",
        "zip_code": "12345",
    },
    employees=[
        {
            "name": "Jane Doe",
            "position": "Chef",
            "email": "jane.doe@example.com"
        },
        {
            "name": "Mike Roe",
            "position": "Waiter",
            "email": "mike.roe@example.com"
        }
    ],
    number_of_seats=50,
    delivery=True,
    website="http://tastybites.com"
)

# Printing the instance
print(restaurant_instance)


name='Tasty Bites' owner=Owner(name='John Doe', email='john.doe@example.com') address=Address(street='123, Flavor Street', city='Tastytown', state='TS', zip_code='12345') employees=[Employee(name='Jane Doe', position='Chef', email='jane.doe@example.com'), Employee(name='Mike Roe', position='Waiter', email='mike.roe@example.com')] number_of_seats=50 delivery=True website=HttpUrl('http://tastybites.com/')


Field Validators

In [15]:
from pydantic import BaseModel, EmailStr, field_validator

class Owner(BaseModel):
    name: str
    email: EmailStr
    
    @field_validator('name')
    @classmethod  # this is a class method because it is called before we create the object.
    def name_must_contain_space(cls, v: str) -> str:
        if ' ' not in v:
            raise ValueError('Owner name must contain a space')
        # we can do some data manipulation now:
        return v.title()

try:
    owner_instance = Owner(name="JohnDoe", email="john.doe@example.com")
except ValueError as e:
    print(e)



1 validation error for Owner
name
  Value error, Owner name must contain a space [type=value_error, input_value='JohnDoe', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


## Model validators - allowing you to create a model before and after field validation

In [49]:
from typing import Any
from pydantic import BaseModel, EmailStr, ValidationError, model_validator

class Owner(BaseModel):
    name: str
    email: EmailStr
    
    @model_validator(mode='before')  # model validator validates a complete data model
    @classmethod
    def check_sensitive_info_omitted(cls, data: Any) -> Any:
        print("running model (class) validator BEFORE validation")
        if isinstance(data, dict):
            if 'password' in data:
                raise ValueError('password should not be included')
            if 'card_number' in data:
                raise ValueError('(before validation) card_number should not be included')
        return data
    
    @model_validator(mode='after')
    def check_name_contains_space(self) -> 'Owner':
        """
        An instance method, which only runs if an instance was created.
        """
        print("running model (instance) validator after validation")
        if ' ' not in self.name:
            raise ValueError('(after validation) Owner name must contain a space')
        return self


print(Owner(name="John Doe", email="john.doe@example.com")) 

running model (class) validator BEFORE validation
running model (instance) validator after validation
name='John Doe' email='john.doe@example.com'


In [24]:
try:
    owner = Owner(name="JohnDoe", email="john.doe@example.com", password="password123")
    print(f"{owner=}")
except ValidationError as e:
    print(f"{e=}") 

running model validator after validation
name='John Doe' email='john.doe@example.com'
e=1 validation error for Owner
  Value error, password should not be included [type=value_error, input_value={'name': 'JohnDoe', 'emai...assword': 'password123'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


In [ ]:
from typing import Any
from pydantic import BaseModel, EmailStr, ValidationError, model_validator

class Owner(BaseModel):
    name: str
    email: EmailStr
    
    @model_validator(mode='before')
    @classmethod
    def check_sensitive_info_omitted(cls, data: Any) -> Any:
        print("running model validator BEFORE validation")
        if isinstance(data, dict):
            if 'password' in data:
                raise ValueError('password should not be included')
            if 'card_number' in data:
                raise ValueError('(before validation) card_number should not be included')
        return data
    
    @model_validator(mode='after')
    def check_name_contains_space(self) -> 'Owner':
        print("running model validator after validation")
        if ' ' not in self.name:
            raise ValueError('(after validation) Owner name must contain a space')
        return self

In [39]:
# print(Owner(name="JohnDoe", email="john.doe@example.com")) 
# try:
#     print(Owner(name="JohnDoe", email="john.doe@example.com", password="swordfish"))
# data_list = []
# error_list = []
validation_errors = []
data = {"name": "JohnDoe", "email":"john.doe@example.com", "password":"password123"}

try:
    owner = Owner(**data)  # name="JohnDoe", email="john.doe@example.com", password="password123")
    print(f"{owner=}")
except ValidationError as e:
    print(f"{e=}") 
    validation_error = {    
        "error": e,
        "data": data
    }
    print(f"{validation_error=}") 
    validation_errors.append(validation_error)

print(f"{validation_errors=}")

running model validator BEFORE validation
e=1 validation error for Owner
  Value error, password should not be included [type=value_error, input_value={'name': 'JohnDoe', 'emai...assword': 'password123'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
validation_error={'error': 1 validation error for Owner
  Value error, password should not be included [type=value_error, input_value={'name': 'JohnDoe', 'emai...assword': 'password123'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error, 'data': {'name': 'JohnDoe', 'email': 'john.doe@example.com', 'password': 'password123'}}
validation_errors=[{'error': 1 validation error for Owner
  Value error, password should not be included [type=value_error, input_value={'name': 'JohnDoe', 'emai...assword': 'password123'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error, 'data': {'name': 'JohnDoe', 'ema

In [25]:
try:
    owner = Owner(name="JohnDoe", email="john.doe@example.com", password="password123")
    print(f"{owner=}")
except ValidationError as e:
    print(f"{e=}") 

running model validator after validation
name='John Doe' email='john.doe@example.com'
e=1 validation error for Owner
  Value error, password should not be included [type=value_error, input_value={'name': 'JohnDoe', 'emai...assword': 'password123'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


In [21]:
try:
    Owner(name="JohnDoe", email="john.doe@example.com")
except ValidationError as e:
    print(e)

1 validation error for Owner
  Value error, Owner name must contain a space [type=value_error, input_value={'name': 'JohnDoe', 'emai... 'john.doe@example.com'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


## Fields - Used to customize and add metadata to fields of models

In [ ]:
from pydantic import BaseModel, Field

class User(BaseModel):
    name: str = Field(default='John Doe')

user = User()
print(user)

In [ ]:
from uuid import uuid4

from pydantic import BaseModel, Field

class User(BaseModel):
    id: int = Field(default_factory=lambda: uuid4().hex)

user = User()
print(user)

Field aliases - For validation and serialization, you can define an alias for a field.

There are three ways to define an alias:

- `Field(..., alias='foo')`
- `Field(..., validation_alias='foo')`
- `Field(..., serialization_alias='foo')`

In [ ]:
from pydantic import BaseModel, Field


class User(BaseModel):
    name: str = Field(..., alias='username')


user = User(username='johndoe')  
print(user)
print(user.model_dump(by_alias=True))

But why would you want to do it? For example if you API and database fields differ - you only need one model

Lets look at field constraints

In [ ]:
from typing import List
from pydantic import BaseModel, Field, EmailStr
from decimal import Decimal

class User(BaseModel):
    username: str = Field(..., min_length=3, max_length=10, pattern=r'^\w+$')
    email: EmailStr = Field(...)
    age: int = Field(..., gt=0, le=120)
    height: float = Field(..., gt=0.0)
    is_active: bool = Field(True)
    balance: Decimal = Field(..., max_digits=10, decimal_places=2)
    favorite_numbers: List[int] = Field(..., min_items=1)


In [ ]:
user_instance = User(
    username="john_doe",
    age=30,
    height=5.9,
    weight=160.5,
    email="john.doe@example.com",
    password="securepassword",
    balance=9999.99,
    favorite_numbers=[1,2,3]
)

print(user_instance)

Computed Fields

In [ ]:
from pydantic import BaseModel, computed_field
from datetime import datetime


class Person(BaseModel):
    name: str
    birth_year: int

    @computed_field
    @property
    def age(self) -> int:
        current_year = datetime.now().year
        return current_year - self.birth_year


print(Person(name="John Doe", birth_year=2000).model_dump())


In [ ]:
from pydantic import BaseModel, ValidationError, field_validator
from datetime import datetime

class Person(BaseModel):
    name: str
    birth_year: int

    @property
    def age(self) -> int:
        current_year = datetime.now().year
        return current_year - self.birth_year

    @field_validator('birth_year')
    @classmethod
    def validate_age(cls, v: int) -> int:
        current_year = datetime.now().year
        if current_year - v < 18:
            raise ValueError('Person must be 18 years or older')
        return v

try:
    print(Person(name="John Doe", birth_year=2006))
except ValidationError as e:
    print(e)


You also also use dataclasses and pydantics valiation logic - dataclasses do not provide that out of the box

In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional
from pydantic import Field, TypeAdapter


@dataclass
class User:
    id: int
    name: str = 'John Doe'
    age: Optional[int] = field(
        default=None,
        metadata=dict(title='The age of the user', description='do not lie!', ge=18),
    )
    height: Optional[int] = Field(None, title='The height in cm', ge=50, le=300)
    friends: List[int] = field(default_factory=lambda: [0])





TypeAdapter

You may have types that are not `BaseModels` that you want to validate data against. Or you may want to validate a `List[SomeModel]`, or dump it to JSON.

For use cases like this, Pydantic provides `TypeAdapter`, which can be used for type validation, serialization, and JSON schema generation without creating a `BaseModel`.


In [ ]:
# Example of using TypeAdapter to get json_schema of the User dataclass
print(TypeAdapter(User).json_schema())

### Strict mode

Pydantic, by default, tries to coerce values to the declared type, converting inputs like the string "123" to integer 123, but this automatic conversion can be undesirable in situations where strict type compliance is required, causing a need for configurations to make Pydantic error out instead.

In [ ]:
from pydantic import BaseModel, ValidationError


class User(BaseModel):
    id: int
    username: str


print(User.model_validate({'id': '42', 'username': 'john_doe'}))  # lax mode
#> id=42 username='john_doe'

In [ ]:
try:
    User.model_validate({'id': '42', 'username': 'john_doe'}, strict=True)  # strict mode
except ValidationError as exc:
    print(exc)

Settings Management 

Pydantic Settings provides optional Pydantic features for loading a settings or config class from environment variables or secrets files.

In [ ]:
!pip install pydantic-settings

In [ ]:
import os
from pydantic import Field
from pydantic_settings import BaseSettings


class Settings(BaseSettings):
    auth_key: str = Field(validation_alias='my_auth_key')  
    api_key: str = Field(alias='my_api_key')  
    

print(Settings().model_dump())

In [ ]:
import os
from pydantic import Field, AliasChoices
from pydantic_settings import BaseSettings

os.environ["AUTH_KEY"] = "test_auth_key"
os.environ["MY_API_KEY"] = "test"
os.environ["ENV2"] = "https://mysuperurl.com"


class Settings(BaseSettings):
    service_name: str = Field(default="default")
    auth_key: str  
    api_key: str = Field(alias='my_api_key')  
    url: str = Field(validation_alias=AliasChoices("env1", "env2"))
     

print(Settings().model_dump())

In [ ]:
import os
from pydantic import Field
from pydantic_settings import BaseSettings, SettingsConfigDict

# Set environment variables with the prefix
os.environ["PRODUCTION_AUTH_KEY"] = "test_auth_key"
os.environ["PRODUCTION_MY_API_KEY"] = "test"
os.environ["PRODUCTION_ENV2"] = "https://mysuperurl.com"


class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_prefix='production_')

    service_name: str = Field(default="default")
    auth_key: str
    api_key: str = Field(alias='my_api_key')
    url: str = Field(validation_alias=AliasChoices("env1", "env2"))

print(Settings().model_dump())


You can also use a .env file

In [ ]:
from pydantic_settings import BaseSettings, SettingsConfigDict


class Settings(BaseSettings):

    model_config = SettingsConfigDict(env_file='.env', env_file_encoding='utf-8')

    service_name: str = Field(default="default")
    auth_key: str  
    api_key: str = Field(alias='my_api_key')  
     

print(Settings().model_dump())

Extra attributes (https://docs.pydantic.dev/2.3/usage/model_config/#extra-attributes)

In [ ]:
from pydantic_settings import BaseSettings, SettingsConfigDict


class Settings(BaseSettings):

    model_config = SettingsConfigDict(env_file='.env', env_file_encoding='utf-8', extra="ignore") # forbid, allow

    service_name: str = Field(default="default")
    auth_key: str  
    api_key: str = Field(alias='my_api_key')  
     

print(Settings().model_dump())